In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
sample_texts = ["Movie was great", "That is an awful lot of waste!"]
sentiments = classifier(sample_texts)
print(sentiments)

In [ ]:
classifier = pipeline("zero-shot-classification")
sample_texts = ["He is a great actor", "I voted yesterday", "We won the world cup last year"]
candidate_labels = ["politics","sports","movies","news"]
results = classifier(sample_texts,candidate_labels=candidate_labels)
print(results)

In [ ]:
generator = pipeline("text-generation")
sample_text = "I was raining quite heavily since "
results = generator(sample_text,num_return_sequences=3,max_length=15)
print(results)

In [ ]:
generator = pipeline("text-generation",model="HuggingFaceTB/SmolLM2-360M")
sample_text = "I was raining quite heavily since "
results = generator(sample_text,max_length=15)
print(results)

In [ ]:
mask_filler = pipeline("fill-mask")
sample_text = "Hello I am <mask>. I am from India"
results = mask_filler(sample_text,top_k=3)
print(results)

In [ ]:
ner = pipeline("ner",grouped_entities=True)
sample_text = "Hello I am Vardhan, a student at IISc Bengaluru"
results = ner(sample_text)
print(results)

## Sentiment Classifier Full Pipeline

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import torch

prompts = ["Movie was great", "That is an awful lot of waste!"]
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokens = tokenizer(prompts,padding=True,truncation=True,return_tensors="pt")
model = AutoModelForSequenceClassification.from_pretrained(model_name)
outputs = model(**tokens)
outputs = F.softmax(outputs.logits,dim=1)
labels = model.config.id2label
answers = [labels[idx.item()] for idx in torch.argmax(outputs,dim=1)]
for i,prompt in enumerate(prompts): print(f"Prompt : {prompt}; Predicted Sentiment : {answers[i]}")

## Simple Training

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
from torch.optim import AdamW

checkpoint = "bert-base-uncased"
prompts = ["I am Vardhan", "I am going to watch a movie"]
targets = [1,1]
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokens = tokenizer(prompts,truncation=True,padding=True,return_tensors="pt")
tokens["labels"] = torch.tensor(targets)
labels = model.config.id2label

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
optimizer = AdamW(model.parameters())
loss = model(**tokens).loss
loss.backward()
optimizer.step()

## Datasets

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

raw_dataset = load_dataset("glue","mrpc")
raw_train_dataset,raw_val_dataset,raw_test_dataset = raw_dataset["train"], raw_dataset["validation"], raw_dataset["test"]
ex_one,ex_two = raw_train_dataset[15],raw_val_dataset[87]

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokens = tokenizer(ex_one["sentence1"],ex_one["sentence2"],truncation=True,padding=True)

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
import torch

checkpoint = "bert-base-uncased"
raw_dataset = load_dataset("glue","mrpc")

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
def tokenize_function(x): return tokenizer(x["sentence1"],x["sentence2"],truncation=True)
tokenized_dataset = raw_dataset.map(tokenize_function,batched=True)

# for batch in tokenized_dataset: ## PSUEDO-CODE
#     batch = {k:v for k,v in batch.items() if k not in ["idx", "sentence1", "sentence2"]} ## We can't process strings, and these are not needed as well
#     batch = data_collator(batch) ## Dynamic Padding

## Fine Tuning

In [ ]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification,DataCollatorWithPadding
from datasets import load_dataset
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

raw_dataset = load_dataset("glue","mrpc")
checkpoint  = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
def tokenize_function(example): return tokenizer(example["sentence1"],example["sentence2"],truncation=True)
tokenized_dataset = raw_dataset.map(tokenize_function,batched=True)
data_collator = DataCollatorWithPadding(tokenizer)

training_args = TrainingArguments("model", eval_strategy="epoch",fp16=True,per_device_train_batch_size=4,gradient_accumulation_steps=4,learning_rate=1e-3,lr_scheduler_type="cosine")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)
def compute_metrics(eval_preds): 
    metric = evaluate.load("glue","mrpc")
    logits,labels = eval_preds
    predictions = np.argmax(logits,axis=-1)
    return metric.compute(predictions=predictions,references=labels)
trainer = Trainer(model,training_args,data_collator,
                  train_dataset=tokenized_dataset["train"],eval_dataset=tokenized_dataset["validation"],
                  processing_class=tokenizer,compute_metrics=compute_metrics)
trainer.train()

predictions = trainer.predict(test_dataset=tokenized_dataset["validation"])